#STURM-Flood

In [1]:
#@title Imports

import random
import os
import sys
import zipfile
import ee

from dataclasses import dataclass
from pathlib import Path
from google.colab import auth, drive

In [ ]:
#@title Google auth + Mount Drive + Optional clone

root_path = "/content/drive/MyDrive/MSc/STURM-fusion"  #@param {type:"string", multiline:true}
mount_point = "/content/drive"  #@param {type:"string"}
clone_repo = False  #@param {type:"boolean"}
reset_export = False  #@param {type:"boolean"}
gee_project = "356457881639"  #@param {type:"string"}

# Always authenticate + mount
auth.authenticate_user()
drive.mount(mount_point)

ee.Initialize(project=gee_project)

repo_url = "https://github.com/TAX2310/STURM-fusion.git"

if clone_repo:
    project_root = os.path.join(root_path, "STURM-fusion")
    if not os.path.exists(project_root):
        !git clone {repo_url} "{project_root}"
else:
    project_root = root_path

sys.path.append(project_root)

from config.config import CFG
cfg = CFG()
cfg.ROOT = Path(project_root)
cfg.DRIVE_ROOT = Path(mount_point) / "MyDrive"
cfg.GEE_PROJECT = gee_project

print("✅ ROOT:", cfg.ROOT)
print("✅ DRIVE_ROOT:", cfg.DRIVE_ROOT)
print("✅ GEE project:", cfg.GEE_PROJECT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ ROOT: /content/drive/MyDrive/MSc/STURM-fusion
✅ DRIVE_ROOT: /content/drive/MyDrive
✅ GEE project: 356457881639


In [3]:
from src.data.sturm_flood import *
from src.pipeline.build_dataset import *
from src.utils.io import *
from src.gee.tasks import has_active_tasks

In [4]:
if reset_export:
    clear_export_folder(cfg)
    
create_dataset_structure(cfg)

🗑️ Deleting 5 files...
Deleted: EMSR279_04TUDELA_07_10_2_1.tif
Deleted: EMSR279_04TUDELA_08_10_1_2.tif
Deleted: EMSR279_04TUDELA_08_10_1_1.tif
Deleted: EMSR279_04TUDELA_08_10_2_2.tif
Deleted: EMSR279_04TUDELA_08_10_2_1.tif
✅ Export folder cleared
✅ Dataset structure created:
 - /content/drive/MyDrive/MSc/STURM-fusion/STURM-flood
 - /content/drive/MyDrive/MSc/STURM-fusion/STURM-fusion-24
 - /content/drive/MyDrive/MSc/STURM-fusion/STURM-fusion-24/Dataset
 - /content/drive/MyDrive/MSc/STURM-fusion/STURM-fusion-24/Dataset/S1
 - /content/drive/MyDrive/MSc/STURM-fusion/STURM-fusion-24/Dataset/S2
 - /content/drive/MyDrive/MSc/STURM-fusion/STURM-fusion-24/Dataset/floodmaps
 - /content/drive/MyDrive/MSc/STURM-fusion/STURM-fusion-24/Dataset/metadata


In [5]:
#@title Download Dataset

download_and_extract(cfg)

✅ Zip already exists or dataset present, skipping download.
✅ Dataset already extracted, skipping unzip.


In [ ]:
if not has_active_tasks():
    print("No active GEE tasks. Starting image collection... 🚀"  )
    images = process_csv(cfg.OLD_S2_METADATA_CSV, cfg)
    print(len(images), "images processed.")
else:
    print("Active GEE tasks detected. Please wait for them to finish starting Image collection. ⏳")



📊 GEE Task Status:
RUNNING   : 0
READY     : 0
COMPLETED : 389
FAILED    : 10
CANCELLED : 1805
➡️ Active tasks: 0
No active GEE tasks. Starting image collection... 🚀
Processing 13/2675: EMSR279_04TUDELA_06_07_1_1.tifAOI area: 1694247.40
Intersection area: 23906.45
Coverage ratio: 0.0141
Skipping EMSR279_04TUDELA_06_07_1_1.tif - S1 does not fully cover AOI
Processing 14/2675: EMSR279_04TUDELA_06_07_1_2.tifAOI area: 1694824.40
Intersection area: 1608564.80
Coverage ratio: 0.9491
Skipping EMSR279_04TUDELA_06_07_1_2.tif - S1 does not fully cover AOI
Processing 16/2675: EMSR279_04TUDELA_06_07_2_2.tifAOI area: 1694805.44
Intersection area: 1261673.84
Coverage ratio: 0.7444
Skipping EMSR279_04TUDELA_06_07_2_2.tif - S1 does not fully cover AOI
Processing 17/2675: EMSR279_04TUDELA_07_10_2_1.tifAOI area: 1697646.61
Intersection area: 1697646.61
Coverage ratio: 1.0000
Processing 18/2675: EMSR279_04TUDELA_08_10_1_1.tifAOI area: 1697626.49
Intersection area: 1697626.49
Coverage ratio: 1.0000
Proce

In [ ]:
if not has_active_tasks():
    print("No active GEE tasks. Starting export... 🚀"  )
    export_all_s1_images(images, cfg)
else:
    print("Active GEE tasks detected. Please wait for them to finish before exporting. ⏳")


✅ All batches submitted


In [ ]:
if not has_active_tasks():
    print("No active GEE tasks. Creating Dataset... 🚀"  )
    create_dataset(cfg)
else:
    print("Active GEE tasks detected. Please wait for them to finish before creating dataset. ⏳")

Copying Matched S2 images... 🚀
Copying 5/5: EMSR279_04TUDELA_08_10_2_2.tif
✅ Copied: 0
⏭️ Skipped existing: 5
⚠️ Missing source: 0
Copying Matched S1 images... 🚀
Copying 5/5: EMSR279_04TUDELA_08_10_2_2.tif
✅ Copied: 2
⏭️ Skipped existing: 3
⚠️ Missing source: 0
Copying Matched Mask images... 🚀
Copying 5/5: EMSR279_04TUDELA_08_10_2_2.tif
✅ Copied: 0
⏭️ Skipped existing: 5
⚠️ Missing source: 0

📊 Validation Results:
Total rows: 5
Missing S2: 0
Missing S1: 0
✅ Dataset complete


In [ ]:
import rasterio
from pathlib import Path


def check_image_shapes(dir1, dir2):
    """
    Compares image shapes between two directories (e.g. S1 vs S2)

    Assumes matching filenames.
    """

    dir1 = Path(dir1)
    dir2 = Path(dir2)

    mismatches = []
    missing = []

    files1 = {f.name for f in dir1.glob("*.tif")}

    for fname in files1:
        path1 = dir1 / fname
        path2 = dir2 / fname

        if not path2.exists():
            missing.append(fname)
            continue

        with rasterio.open(path1) as src1, rasterio.open(path2) as src2:
            shape1 = (src1.count, src1.height, src1.width)
            shape2 = (src2.count, src2.height, src2.width)

        if shape1 != shape2:
            mismatches.append((fname, shape1, shape2))

    print(f"\n📊 Shape Check Results:")
    print(f"Checked: {len(files1)} files")
    print(f"Missing in dir2: {len(missing)}")
    print(f"Mismatched shapes: {len(mismatches)}")

    return {
        "missing": missing,
        "mismatches": mismatches
    }

result = check_image_shapes(cfg.NEW_S1_PATH, cfg.NEW_S2_PATH)

In [ ]:
import pandas as pd


def get_max_time_difference_with_row(csv_path):
    df = pd.read_csv(csv_path)

    df["floodmap_date"] = pd.to_datetime(df["floodmap_date"], dayfirst=True, errors="coerce")
    df["sentinel_timestamp"] = pd.to_datetime(df["sentinel_timestamp"], dayfirst=True, errors="coerce")

    df = df.dropna(subset=["floodmap_date", "sentinel_timestamp"])

    df["time_diff_hours"] = (
        (df["sentinel_timestamp"] - df["floodmap_date"])
        .abs()
        .dt.total_seconds() / 3600
    )

    idx = df["time_diff_hours"].idxmax()
    row = df.loc[idx]

    print(f"⏱️ Max time difference: {row['time_diff_hours']:.2f} hours")
    print(row)

    return row

max_diff = get_max_time_difference_with_row(cfg.NEW_METADATA_CSV)

print(max_diff)

/tmp/ipykernel_51842/1766793265.py:7: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["floodmap_date"] = pd.to_datetime(df["floodmap_date"], dayfirst=True, errors="coerce")


KeyError: 'sentinel_timestamp'

In [ ]:
import pandas as pd


def count_over_72h(csv_path):
    """
    Counts number of samples where time difference > 72 hours
    """

    df = pd.read_csv(csv_path)

    df["floodmap_date"] = pd.to_datetime(df["floodmap_date"], dayfirst=True, errors="coerce")
    df["sentinel_timestamp"] = pd.to_datetime(df["sentinel_timestamp"], dayfirst=True, errors="coerce")

    df = df.dropna(subset=["floodmap_date", "sentinel_timestamp"])

    time_diff_hours = (
        (df["sentinel_timestamp"] - df["floodmap_date"])
        .abs()
        .dt.total_seconds() / 3600
    )

    count = (time_diff_hours > 72).sum()

    print(f"⏱️ Samples >72h: {count} / {len(df)}")

    return count

count_over_72h(cfg.NEW_METADATA_CSV)

/tmp/ipykernel_35298/997280217.py:11: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["floodmap_date"] = pd.to_datetime(df["floodmap_date"], dayfirst=True, errors="coerce")


KeyError: 'sentinel_timestamp'

In [ ]:
def cancel_all_tasks(verbose=True):
    """
    Cancels all READY or RUNNING Earth Engine tasks.
    """

    tasks = ee.batch.Task.list()

    cancelled = 0
    skipped = 0

    for t in tasks:
        status = t.status()
        state = status['state']
        desc = status.get('description', 'N/A')

        if state in ['READY', 'RUNNING']:
            t.cancel()
            cancelled += 1
            if verbose:
                print(f"❌ Cancelled: {desc} ({state})")
        else:
            skipped += 1
            if verbose:
                print(f"⏭️ Skipped: {desc} ({state})")

    print("\n📊 Summary:")
    print(f"Cancelled: {cancelled}")
    print(f"Skipped (already done): {skipped}")

#cancel_all_tasks()